In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

os.makedirs('images', exist_ok=True)

print("Libraries loaded.")

Libraries loaded.


In [3]:
campaigns = pd.read_csv('data/campaigns.csv')
platforms = pd.read_csv('data/platforms.csv')
campaign_types = pd.read_csv('data/campaign_types.csv')

print("Data loaded.")

Data loaded.


In [4]:
# Create copy for analysis
df = campaigns.copy()

# Calculate KPIs
df['CTR'] = df['Clicks'] / df['Impressions']
df['CPC'] = df['Cost'] / df['Clicks'].replace(0, np.nan)
df['CPM'] = (df['Cost'] / df['Impressions']) * 1000
df['CVR'] = df['Conversions'] / df['Clicks'].replace(0, np.nan)
df['CPA'] = df['Cost'] / df['Conversions'].replace(0, np.nan)
df['ROAS'] = df['Revenue'] / df['Cost'].replace(0, np.nan)

# Calculate benchmarks
median_ctr = df['CTR'].median()
median_cvr = df['CVR'].median()
median_cpa = df['CPA'].median()
median_roas = df['ROAS'].median()

print("Benchmarks calculated.")
print("-" * 40)
print(f"Median CTR:  {median_ctr*100:.2f}%")
print(f"Median CVR:  {median_cvr*100:.2f}%")
print(f"Median CPA:  ${median_cpa:.2f}")
print(f"Median ROAS: {median_roas:.2f}x")

Benchmarks calculated.
----------------------------------------
Median CTR:  1.70%
Median CVR:  5.25%
Median CPA:  $37.62
Median ROAS: 2.84x


In [5]:
def calculate_health_score(row):
    """Calculate campaign health score (0-100)"""
    score = 0
    
    # CTR (20 points)
    if row['CTR'] > median_ctr * 1.5:
        score += 20
    elif row['CTR'] > median_ctr:
        score += 15
    elif row['CTR'] > median_ctr * 0.5:
        score += 10
    else:
        score += 5
    
    # CVR (20 points)
    if row['CVR'] > median_cvr * 1.5:
        score += 20
    elif row['CVR'] > median_cvr:
        score += 15
    elif row['CVR'] > median_cvr * 0.5:
        score += 10
    else:
        score += 5
    
    # CPA (20 points)
    if row['CPA'] < median_cpa * 0.5:
        score += 20
    elif row['CPA'] < median_cpa:
        score += 15
    elif row['CPA'] < median_cpa * 1.5:
        score += 10
    else:
        score += 5
    
    # ROAS (20 points)
    if row['ROAS'] > median_roas * 1.5:
        score += 20
    elif row['ROAS'] > median_roas:
        score += 15
    elif row['ROAS'] > 1.0:
        score += 10
    else:
        score += 5
    
    # Volume (20 points)
    if row['Conversions'] > 100:
        score += 20
    elif row['Conversions'] > 50:
        score += 15
    elif row['Conversions'] > 20:
        score += 10
    else:
        score += 5
    
    return min(100, score)

df['Health_Score'] = df.apply(calculate_health_score, axis=1)

print("Health scores calculated.")
print(f"Mean Health Score: {df['Health_Score'].mean():.1f}")
print(f"Median Health Score: {df['Health_Score'].median():.0f}")

Health scores calculated.
Mean Health Score: 68.6
Median Health Score: 70


In [6]:
def classify_campaign(row):
    """Classify campaign into recommendation categories"""
    
    # Flag outliers (high ROAS with limited volume)
    is_outlier = row['ROAS'] > 10 and row['Conversions'] < 100
    
    # SCALE: High health, good ROAS, sufficient volume, not outlier
    if (row['Health_Score'] >= 70 and 
        row['ROAS'] > 2.0 and 
        row['Conversions'] > 50 and
        not is_outlier):
        return 'Scale'
    
    # RETARGET: High engagement, low conversion, sufficient impressions
    if (row['CTR'] > median_ctr and 
        row['CVR'] < median_cvr and 
        row['Impressions'] > 5000 and
        row['Clicks'] > 100 and
        row['ROAS'] < 2.0):
        return 'Retarget'
    
    # PAUSE: Poor performance across multiple dimensions
    if (row['ROAS'] < 1.5 or 
        row['Health_Score'] < 40 or
        (row['CTR'] < median_ctr * 0.5 and row['CVR'] < median_cvr * 0.5)):
        return 'Pause'
    
    # OPTIMIZE: Everything else
    return 'Optimize'

df['Decision'] = df.apply(classify_campaign, axis=1)

print("Campaigns classified.")

Campaigns classified.


In [7]:
decision_counts = df['Decision'].value_counts()

print("Campaign Decisions")
print("-" * 40)
for decision, count in decision_counts.items():
    pct = count / len(df) * 100
    print(f"{decision}: {count:,} ({pct:.1f}%)")

Campaign Decisions
----------------------------------------
Scale: 5,115 (51.1%)
Optimize: 3,663 (36.6%)
Pause: 1,008 (10.1%)
Retarget: 214 (2.1%)


In [8]:
# Identify scalable campaigns
scalable_criteria = (
    (df['Decision'] == 'Scale') &
    (df['ROAS'] < 10) &
    (df['Conversions'] > 50) &
    (df['Health_Score'] >= 70)
)

scalable = df[scalable_criteria].copy()

print("Scalable Campaigns")
print("-" * 40)
print(f"Total: {len(scalable)} campaigns")
print(f"Total Spend: ${scalable['Cost'].sum():,.2f}")
print(f"Total Revenue: ${scalable['Revenue'].sum():,.2f}")
print(f"Average ROAS: {scalable['ROAS'].mean():.2f}x")
print(f"Average ROI: {scalable['ROI'].mean():.1f}%")
print(f"Average Health Score: {scalable['Health_Score'].mean():.1f}")

Scalable Campaigns
----------------------------------------
Total: 5059 campaigns
Total Spend: $386,772,637.67
Total Revenue: $1,198,486,108.74
Average ROAS: 3.02x
Average ROI: 202.3%
Average Health Score: 76.9


In [9]:
top_scalable = scalable.nlargest(10, 'ROAS')[['CampaignID', 'PlatformID', 'CampaignTypeID', 
                                               'ROAS', 'ROI', 'CTR', 'CVR', 'CPA', 
                                               'Conversions', 'Cost', 'Health_Score']]

print("Top 10 Scalable Campaigns")
print("-" * 40)
print(top_scalable.to_string(index=False))

Top 10 Scalable Campaigns
----------------------------------------
CampaignID  PlatformID  CampaignTypeID     ROAS    ROI      CTR      CVR       CPA  Conversions     Cost  Health_Score
 CAM004207           6               6 9.944843 894.48 0.007546 0.039344 24.042834          554 13319.73            70
 CAM002930           3               3 9.162730 816.27 0.017828 0.060075 25.943069         2649 68723.19            85
 CAM003633           5               4 9.122155 812.22 0.013968 0.045819 24.969550         1022 25518.88            75
 CAM007705           5               2 9.054570 805.46 0.020591 0.062662 17.677256          871 15396.89            90
 CAM009227           2               3 8.920552 792.06 0.029829 0.060672 23.337225          973 22707.12            90
 CAM002133           3               4 8.717637 771.76 0.018748 0.047199 42.430494         1194 50662.01            75
 CAM007885           3               5 8.265839 726.58 0.016213 0.044891 31.306501          866 2711

In [10]:
# Identify outlier campaigns
outlier_criteria = (
    (df['ROAS'] > 10) &
    (df['Conversions'] < 100)
)

outliers = df[outlier_criteria].copy()

print("Outlier Campaigns (Need Validation)")
print("-" * 40)
print(f"Total: {len(outliers)} campaigns")
print(f"Average ROAS: {outliers['ROAS'].mean():.2f}x")
print(f"Average Conversions: {outliers['Conversions'].mean():.1f}")
print(f"Average Cost: ${outliers['Cost'].mean():,.2f}")

print("\nSample Outlier Campaigns:")
print(outliers[['CampaignID', 'ROAS', 'ROI', 'Conversions', 'Cost', 'Health_Score']].head(10).to_string(index=False))

Outlier Campaigns (Need Validation)
----------------------------------------
Total: 9 campaigns
Average ROAS: 19.91x
Average Conversions: 52.3
Average Cost: $2,876.23

Sample Outlier Campaigns:
CampaignID      ROAS     ROI  Conversions    Cost  Health_Score
 CAM000723 11.446938 1044.69           16 1154.68            50
 CAM002072 18.747229 1500.00           66 4583.48            60
 CAM002167 23.932785 1500.00           22  558.06            60
 CAM003922 17.734726 1500.00           48 4144.59            50
 CAM005180 30.691280 1500.00           54  997.44            85
 CAM005594 16.952009 1500.00           63 6794.17            65
 CAM006213 13.436743 1243.68           49 1291.88            60
 CAM009575 33.268204 1500.00           87 2787.06            80
 CAM009820 12.950108 1195.01           66 3574.74            70


In [11]:
# Aggregate by platform
platform_allocation = scalable.groupby('PlatformID').agg({
    'Cost': 'sum',
    'Revenue': 'sum',
    'ROAS': 'mean',
    'ROI': 'mean',
    'CampaignID': 'count',
    'Health_Score': 'mean'
}).round(2)

platform_allocation.columns = ['Total_Spend', 'Total_Revenue', 'Avg_ROAS', 'Avg_ROI', 'Campaigns', 'Avg_Health']

# Merge with platform names
platform_allocation = platform_allocation.merge(platforms[['PlatformID', 'PlatformName']], on='PlatformID')
platform_allocation = platform_allocation.sort_values('Avg_ROI', ascending=False)

print("Platform Allocation Summary")
print("-" * 40)
print(platform_allocation[['PlatformName', 'Campaigns', 'Total_Spend', 'Avg_ROI']].to_string(index=False))

Platform Allocation Summary
----------------------------------------
PlatformName  Campaigns  Total_Spend  Avg_ROI
    LinkedIn          5    332041.45   287.31
  Google Ads        994 167148748.54   222.27
      TikTok       1097  76065956.17   211.96
   Instagram        893  61474988.80   204.37
     YouTube        555  44446784.00   195.65
    Facebook        812  25835265.43   187.48
 X (Twitter)        703  11468853.28   178.19


In [12]:
# Aggregate by campaign type
type_allocation = scalable.groupby('CampaignTypeID').agg({
    'Cost': 'sum',
    'Revenue': 'sum',
    'ROAS': 'mean',
    'ROI': 'mean',
    'CampaignID': 'count',
    'Health_Score': 'mean'
}).round(2)

type_allocation.columns = ['Total_Spend', 'Total_Revenue', 'Avg_ROAS', 'Avg_ROI', 'Campaigns', 'Avg_Health']

# Merge with campaign type names
type_allocation = type_allocation.merge(campaign_types[['CampaignTypeID', 'CampaignType']], on='CampaignTypeID')
type_allocation = type_allocation.sort_values('Avg_ROI', ascending=False)

print("Campaign Type Allocation Summary")
print("-" * 40)
print(type_allocation[['CampaignType', 'Campaigns', 'Total_Spend', 'Avg_ROI']].to_string(index=False))

Campaign Type Allocation Summary
----------------------------------------
CampaignType  Campaigns  Total_Spend  Avg_ROI
     Display        155   7163984.38   230.87
       Email        589  41497253.53   210.82
      Social        895  68178091.65   206.16
       Video       1065  82453208.17   200.90
 Retargeting       1175  87901091.18   198.57
      Search       1180  99579008.76   196.38


In [13]:
def calculate_incremental_score(row):
    """Score campaigns for incremental budget allocation"""
    score = 0
    
    # ROAS contribution (30 points)
    if row['ROAS'] > 5.0:
        score += 30
    elif row['ROAS'] > 3.0:
        score += 20
    elif row['ROAS'] > 2.0:
        score += 10
    
    # CTR contribution (15 points)
    if row['CTR'] > median_ctr * 1.5:
        score += 15
    elif row['CTR'] > median_ctr:
        score += 10
    
    # CVR contribution (15 points)
    if row['CVR'] > median_cvr * 1.5:
        score += 15
    elif row['CVR'] > median_cvr:
        score += 10
    
    # CPA contribution (15 points)
    if row['CPA'] < median_cpa * 0.5:
        score += 15
    elif row['CPA'] < median_cpa:
        score += 10
    
    # Volume contribution (15 points)
    if row['Conversions'] > 200:
        score += 15
    elif row['Conversions'] > 100:
        score += 10
    elif row['Conversions'] > 50:
        score += 5
    
    # Health score contribution (10 points)
    score += row['Health_Score'] / 10
    
    # Outlier penalty
    if row['ROAS'] > 10 and row['Conversions'] < 100:
        score -= 20
    
    return max(0, min(100, score))

df['Incremental_Score'] = df.apply(calculate_incremental_score, axis=1)

print("Incremental scores calculated.")

Incremental scores calculated.


In [14]:
# Filter for scale campaigns only
scale_candidates = df[df['Decision'] == 'Scale'].copy()
scale_candidates = scale_candidates.sort_values('Incremental_Score', ascending=False)

print("Top 10 Incremental Budget Candidates")
print("-" * 40)

top_candidates = scale_candidates.head(10)[['CampaignID', 'PlatformID', 'CampaignTypeID', 
                                              'ROAS', 'ROI', 'CTR', 'CVR', 'CPA', 
                                              'Conversions', 'Health_Score', 'Incremental_Score']]

# Add platform and type names
top_candidates = top_candidates.merge(platforms[['PlatformID', 'PlatformName']], on='PlatformID')
top_candidates = top_candidates.merge(campaign_types[['CampaignTypeID', 'CampaignType']], on='CampaignTypeID')

print(top_candidates[['CampaignID', 'PlatformName', 'CampaignType', 
                       'ROAS', 'ROI', 'Conversions', 'Incremental_Score']].to_string(index=False))

Top 10 Incremental Budget Candidates
----------------------------------------
CampaignID PlatformName CampaignType      ROAS     ROI  Conversions  Incremental_Score
 CAM000409       TikTok       Search 28.107767 1500.00         4751               94.5
 CAM007964       TikTok       Search  3.110565  211.06         2742               89.5
 CAM002234       TikTok       Search  3.040087  204.01         4261               89.5
 CAM009926      YouTube       Search 11.758210 1075.82         7507               89.0
 CAM009227       TikTok        Video  8.920552  792.06          973               89.0
 CAM003382       TikTok       Search  7.228534  622.85         7167               89.0
 CAM007920       TikTok       Search 17.709246 1500.00         3558               89.0
 CAM005492   Google Ads       Search  6.819728  581.97         2649               89.0
 CAM001032   Google Ads  Retargeting  6.631819  563.18         1156               89.0
 CAM003620    Instagram        Video 17.576478 1500.

In [16]:
# Best candidate for incremental budget
best = top_candidates.iloc[0]

print("Recommended Next $1,000 Investment")
print("-" * 40)
print(f"Campaign: {best['CampaignID']}")
print(f"Platform: {best['PlatformName']}")
print(f"Campaign Type: {best['CampaignType']}")
print(f"Current ROAS: {best['ROAS']:.2f}x")
print(f"Current ROI: {best['ROI']:.1f}%")
print(f"Conversions: {best['Conversions']:.0f}")
print(f"Incremental Score: {best['Incremental_Score']:.0f}/100")

Recommended Next $1,000 Investment
----------------------------------------
Campaign: CAM000409
Platform: TikTok
Campaign Type: Search
Current ROAS: 28.11x
Current ROI: 1500.0%
Conversions: 4751
Incremental Score: 94/100


In [18]:
print("Budget Allocation Summary")
print("-" * 40)

print("\nDecisions:")
for decision, count in decision_counts.items():
    pct = count / len(df) * 100
    print(f"  {decision}: {count:,} ({pct:.1f}%)")

print("\nScalable Campaigns:")
print(f"  Total: {len(scalable)} campaigns")
print(f"  Total Spend: ${scalable['Cost'].sum():,.2f}")
print(f"  Total Revenue: ${scalable['Revenue'].sum():,.2f}")
print(f"  Average ROAS: {scalable['ROAS'].mean():.2f}x")
print(f"  Average ROI: {scalable['ROI'].mean():.1f}%")

print("\nOutlier Campaigns (Need Validation):")
print(f"  Total: {len(outliers)} campaigns")
print(f"  Average ROAS: {outliers['ROAS'].mean():.2f}x")

print("\nTop Incremental Candidate:")
print(f"  Campaign: {best['CampaignID']}")
print(f"  Platform: {best['PlatformName']}")
print(f"  Type: {best['CampaignType']}")
print(f"  ROAS: {best['ROAS']:.2f}x")
print(f"  ROI: {best['ROI']:.1f}%")
print(f"  Incremental Score: {best['Incremental_Score']:.0f}/100")

Budget Allocation Summary
----------------------------------------

Decisions:
  Scale: 5,115 (51.1%)
  Optimize: 3,663 (36.6%)
  Pause: 1,008 (10.1%)
  Retarget: 214 (2.1%)

Scalable Campaigns:
  Total: 5059 campaigns
  Total Spend: $386,772,637.67
  Total Revenue: $1,198,486,108.74
  Average ROAS: 3.02x
  Average ROI: 202.3%

Outlier Campaigns (Need Validation):
  Total: 9 campaigns
  Average ROAS: 19.91x

Top Incremental Candidate:
  Campaign: CAM000409
  Platform: TikTok
  Type: Search
  ROAS: 28.11x
  ROI: 1500.0%
  Incremental Score: 94/100


In [24]:

print("FINAL AUTHORITATIVE SUMMARY")
print("=" * 50)

print("\nCAMPAIGN DECISIONS (Source of Truth):")
print("-" * 40)
for decision, count in decision_counts.items():
    pct = count / len(df) * 100
    print(f"  {decision}: {count:,} ({pct:.1f}%)")

print("\nPLATFORM RANKINGS (by ROI):")
print("-" * 40)

# Calculate platform rankings from current data
platform_rank = df.groupby('PlatformID')['ROI'].mean().reset_index()
platform_rank = platform_rank.merge(platforms[['PlatformID', 'PlatformName']], on='PlatformID')
platform_rank = platform_rank.sort_values('ROI', ascending=False)

for i, row in platform_rank.iterrows():
    print(f"  {i+1}. {row['PlatformName']}: {row['ROI']:.1f}%")

print("\nCAMPAIGN TYPE RANKINGS (by CVR):")
print("-" * 40)

# Calculate campaign type rankings from current data
type_rank = df.groupby('CampaignTypeID')['CVR'].mean().reset_index()
type_rank = type_rank.merge(campaign_types[['CampaignTypeID', 'CampaignType']], on='CampaignTypeID')
type_rank = type_rank.sort_values('CVR', ascending=False)

for i, row in type_rank.iterrows():
    print(f"  {i+1}. {row['CampaignType']}: {row['CVR']*100:.2f}%")

print("\nTOP INCREMENTAL CANDIDATE:")
print("-" * 40)

# Get top candidate from earlier in this notebook
best = top_candidates.iloc[0]
print(f"  Campaign: {best['CampaignID']}")
print(f"  Platform: {best['PlatformName']}")
print(f"  Type: {best['CampaignType']}")
print(f"  ROAS: {best['ROAS']:.2f}x")
print(f"  Incremental Score: {best['Incremental_Score']:.0f}/100")

print("\nRETARGETING OPPORTUNITIES:")
print("-" * 40)

# Calculate retargeting candidates from current data
retargeting_criteria = (
    (df['CTR'] > median_ctr) &
    (df['CVR'] < median_cvr) &
    (df['Impressions'] > 5000) &
    (df['Clicks'] > 100) &
    (df['ROAS'] < 2.0)
)

retargeting_candidates = df[retargeting_criteria].copy()

print(f"  Campaigns: {len(retargeting_candidates)}")
print(f"  Non-converting clicks: {retargeting_candidates['Clicks'].sum() - retargeting_candidates['Conversions'].sum():,.0f}")



FINAL AUTHORITATIVE SUMMARY

CAMPAIGN DECISIONS (Source of Truth):
----------------------------------------
  Scale: 5,115 (51.1%)
  Optimize: 3,663 (36.6%)
  Pause: 1,008 (10.1%)
  Retarget: 214 (2.1%)

PLATFORM RANKINGS (by ROI):
----------------------------------------
  1. Google Ads: 206.5%
  2. TikTok: 192.5%
  3. Instagram: 184.4%
  4. YouTube: 177.6%
  5. Facebook: 167.2%
  6. X (Twitter): 157.6%
  7. LinkedIn: 147.8%

CAMPAIGN TYPE RANKINGS (by CVR):
----------------------------------------
  1. Search: 6.40%
  2. Retargeting: 6.04%
  3. Video: 5.66%
  4. Social: 5.02%
  5. Email: 4.28%
  6. Display: 3.59%

TOP INCREMENTAL CANDIDATE:
----------------------------------------
  Campaign: CAM000409
  Platform: TikTok
  Type: Search
  ROAS: 28.11x
  Incremental Score: 94/100

RETARGETING OPPORTUNITIES:
----------------------------------------
  Campaigns: 214
  Non-converting clicks: 8,530,537
